In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [3]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26.4M/26.4M [00:09<00:00, 2.69MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 146kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.42M/4.42M [00:01<00:00, 2.21MB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<00:00, 5.15MB/s]


In [4]:
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(),
                           v2.ToDtype(torch.float32, scale = True)
                           ])
)

In [5]:
train_dataloader = DataLoader(training_data, batch_size = 64)
test_dataloader = DataLoader(test_data, batch_size = 64 )

In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [8]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch%100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0,0

    with torch.no_grad():
        for X,y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1)==y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size
        print(f"test error: \n accuracy: {(100*correct):>0.1f}%, avg loss: {test_loss:>8f} \n")

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

epochs = 10
for t in range(epochs):
    print(f"epoch {t+1}\n--------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

epoch 1
--------------
loss: 2.307665 [   64/60000]
loss: 2.294895 [ 6464/60000]
loss: 2.270415 [12864/60000]
loss: 2.255427 [19264/60000]
loss: 2.255658 [25664/60000]
loss: 2.224855 [32064/60000]
loss: 2.230095 [38464/60000]
loss: 2.194808 [44864/60000]
loss: 2.182241 [51264/60000]
loss: 2.162242 [57664/60000]
test error: 
 accuracy: 40.3%, avg loss: 2.148255 

epoch 2
--------------
loss: 2.152719 [   64/60000]
loss: 2.143584 [ 6464/60000]
loss: 2.079543 [12864/60000]
loss: 2.091565 [19264/60000]
loss: 2.046604 [25664/60000]
loss: 1.991051 [32064/60000]
loss: 2.010064 [38464/60000]
loss: 1.929233 [44864/60000]
loss: 1.918960 [51264/60000]
loss: 1.858451 [57664/60000]
test error: 
 accuracy: 55.6%, avg loss: 1.852036 

epoch 3
--------------
loss: 1.878880 [   64/60000]
loss: 1.852437 [ 6464/60000]
loss: 1.732084 [12864/60000]
loss: 1.766735 [19264/60000]
loss: 1.660038 [25664/60000]
loss: 1.626124 [32064/60000]
loss: 1.633403 [38464/60000]
loss: 1.545993 [44864/60000]
loss: 1.558653 